In [62]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool, InjectedToolArg
from typing import Annotated
from dotenv import load_dotenv
import json
import requests

In [63]:
load_dotenv()

True

In [64]:
##create tools
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
    """This function fetches the conversion factor between a base and a target currency"""

    url = f'https://v6.exchangerate-api.com/v6/76e34ed7df8acd5b3e5b4bf9/pair/{base_currency}/{target_currency}'

    response = requests.get(url)
    return response.json()

##use injected tool argument so that llm doesn't select the conversion rate by itself
@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
    """given a base currency value and the conversion rate, convert the base currency to the target currency"""

    return base_currency_value * conversion_rate

In [75]:
conversion_factor = get_conversion_factor.invoke({'base_currency': 'USD', 'target_currency': 'INR'})['conversion_rate']
conversion_factor

95.9069

In [66]:
convert.invoke({'base_currency_value': 10, 'conversion_rate': conversion_factor})

959.069

In [67]:
##tool binding

llm = ChatOpenAI()

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [68]:
messages = []

query = HumanMessage('What is the conversion factor between USD and INR, and based on that can you convert 10USD to INR')
messages.append(query)
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10USD to INR', additional_kwargs={}, response_metadata={})]

In [69]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [70]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_i5DLbbv0hDvzSwQHaIZ22CPL',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_currency_value': 10},
  'id': 'call_EaINzag4dkiALUHNyeWQgG44',
  'type': 'tool_call'}]

In [71]:
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message_1 = get_conversion_factor.invoke(tool_call)
        ##fetch value of conversion rate
        conversion_rate = json.loads(tool_message_1.content)['conversion_rate']
        ##append message to message list
        messages.append(tool_message_1)

    ##execute 2nd tool using conversion rate from the previous
    if tool_call['name'] == 'convert':
        tool_call['args']['conversion_rate'] = conversion_rate
        tool_message_2 = convert.invoke(tool_call)
        messages.append(tool_message_2)

In [72]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR, and based on that can you convert 10USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 119, 'total_tokens': 171, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-EYYXj15k7MS6l9ETAwI2YOap7qrZE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a135d6-f825-7111-936a-79271b8a3f3d-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'IN

In [73]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and INR is 95.9069. Based on this, 10 USD is equivalent to 959.069 INR.'